In [133]:
import pandas as pd
data = pd.read_csv("data/Food_Delivery.csv")
data.head()

,order_id,customer_id,restaurant,city,order_date,delivery_time_minutes,item,quantity,price,discount,payment_method,rating
0,1,1102,Kebab House,alex,2024-01-01 00:00:00,40.0,Kebab,4,265,13,Wallet,1.0
1,2,1435,Sushi World,alex,2024-01-01 01:00:00,30.0,Pasta,2,241,5,Card,3.0
2,3,1860,Burger Town,Cairo,2024-01-01 02:00:00,NaN,Pasta,3,240,13,Wallet,2.0
3,4,1270,Sushi World,cairo,2024-01-01 03:00:00,43.0,Kebab,3,269,16,Wallet,2.0
4,5,1106,Burger Town,Cairo,2024-01-01 04:00:00,12.0,Pizza,1,129,17,Cash,2.0


# Data Quality Assessment

In [134]:
data.isnull().sum()

order_id                   0
customer_id                0
restaurant                 0
city                     103
order_date                 0
delivery_time_minutes    107
item                       0
quantity                   0
price                      0
discount                   0
payment_method             0
rating                   105
dtype: int64

In [135]:
data.duplicated().sum()

np.int64(50)

In [136]:
for col in ["restaurant", "city", "item", "payment_method"]:
    print(data[col].unique())

<StringArray>
['Kebab House', 'Sushi World', 'Burger Town', 'Pizza Hub', 'Pasta Point']
Length: 5, dtype: str
<StringArray>
['alex', 'Cairo', 'cairo', nan, 'Alex', 'Giza', 'CAIRO']
Length: 7, dtype: str
<StringArray>
['Kebab', 'Pasta', 'Pizza', 'Burger', 'Sushi']
Length: 5, dtype: str
<StringArray>
['Wallet', 'Card', 'Cash']
Length: 3, dtype: str


In [137]:
def outliers():
    outliers = []
    for col in ["delivery_time_minutes", "quantity", "price", "discount", "rating"]:
        Q1 = data[col].quantile(0.25)
        Q3 = data[col].quantile(0.75)
        IQR = Q3 - Q1
        
        outliers.append(f"{col}: {data[(data[col] < Q1 - 1.5 * IQR) | (data[col] > Q3 + 1.5 * IQR)][col].tolist()}")
    print(outliers)
outliers()

['delivery_time_minutes: [0.0, 69.0, 259.0, 208.0, 0.0, 193.0, 258.0, 73.0, 270.0, 72.0, 0.0, -1.0, 0.0, 233.0, 292.0, 178.0, 245.0, 219.0, 0.0, 0.0, -3.0, -3.0, 71.0, 72.0]', 'quantity: [39, 22, 30, 41, 35, 25, 34, 29, 40, 28]', 'price: []', 'discount: []', 'rating: []']


- There are missing values in "city", "delivery_time_minutes", and "rating" columns
- There are 50 duplicated columns
- There are outliers in "delivery_time_minutes" and "quantity"

In [138]:
print(data.duplicated().sum())
data = data.drop_duplicates()
data.duplicated().sum()

50


np.int64(0)

I decided to delete duplicate, becuase each order is specialist by the order id

In [139]:
data.fillna({
    "city": data["city"].mode()[0],
    "delivery_time_minutes": data["delivery_time_minutes"].median(),
    "rating": data["rating"].mean()
}, inplace=True)
data.isnull().sum()

order_id                 0
customer_id              0
restaurant               0
city                     0
order_date               0
delivery_time_minutes    0
item                     0
quantity                 0
price                    0
discount                 0
payment_method           0
rating                   0
dtype: int64

I chose to replace missing values in "city" with mode because it's a categorical column, replace that in "delivery_time_minutes" with median because it has outliers, and that in "rating" with mean because it's a numerical column and balance

In [140]:
data["city"].unique()

<StringArray>
['alex', 'Cairo', 'cairo', 'CAIRO', 'Alex', 'Giza']
Length: 6, dtype: str

In [141]:
data["city"] = data["city"].str.strip().str.capitalize()
data["city"].unique()

<StringArray>
['Alex', 'Cairo', 'Giza']
Length: 3, dtype: str

In [142]:
data.dtypes

order_id                   int64
customer_id                int64
restaurant                   str
city                         str
order_date                   str
delivery_time_minutes    float64
item                         str
quantity                   int64
price                      int64
discount                   int64
payment_method               str
rating                   float64
dtype: object

In [143]:
data["order_date"] = pd.to_datetime(data["order_date"])
data.dtypes

order_id                          int64
customer_id                       int64
restaurant                          str
city                                str
order_date               datetime64[us]
delivery_time_minutes           float64
item                                str
quantity                          int64
price                             int64
discount                          int64
payment_method                      str
rating                          float64
dtype: object

In [144]:
data.describe()

,order_id,customer_id,order_date,delivery_time_minutes,quantity,price,discount,rating
count,2000.000000,2000.000000,2000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,1000.500000,1498.918500,2024-02-11 15:30:00,35.308500,2.657500,172.035000,14.521500,2.981579
min,1.000000,1000.000000,2024-01-01 00:00:00,-3.000000,1.000000,50.000000,0.000000,1.000000
25%,500.750000,1232.000000,2024-01-21 19:45:00,27.000000,1.000000,111.000000,7.000000,2.000000
50%,1000.500000,1503.000000,2024-02-11 15:30:00,35.000000,3.000000,170.000000,15.000000,3.000000
75%,1500.250000,1761.000000,2024-03-03 11:15:00,42.000000,4.000000,233.000000,22.000000,4.000000
max,2000.000000,1998.000000,2024-03-24 07:00:00,292.000000,41.000000,299.000000,29.000000,5.000000
std,577.494589,294.301136,NaN,18.620249,2.425311,71.491428,8.475698,1.375908


In [145]:
data[data["delivery_time_minutes"] <= 0]

,order_id,customer_id,restaurant,city,order_date,delivery_time_minutes,item,quantity,price,discount,payment_method,rating
30,31,1276,Sushi World,Alex,2024-01-02 06:00:00,0.0,Sushi,4,200,7,Card,5.0
332,333,1931,Pizza Hub,Alex,2024-01-14 20:00:00,0.0,Kebab,3,52,3,Cash,2.0
911,912,1133,Kebab House,Cairo,2024-02-07 23:00:00,0.0,Burger,2,93,19,Cash,2.0
1076,1077,1388,Kebab House,Cairo,2024-02-14 20:00:00,-1.0,Kebab,3,94,14,Cash,5.0
1085,1086,1918,Kebab House,Cairo,2024-02-15 05:00:00,0.0,Sushi,4,130,25,Wallet,3.0
1495,1496,1516,Sushi World,Cairo,2024-03-03 07:00:00,0.0,Pizza,4,228,3,Card,3.0
1513,1514,1156,Pizza Hub,Alex,2024-03-04 01:00:00,0.0,Burger,4,149,1,Wallet,2.0
1603,1604,1802,Pasta Point,Alex,2024-03-07 19:00:00,-3.0,Kebab,2,153,1,Wallet,1.0
1657,1658,1140,Kebab House,Alex,2024-03-10 01:00:00,-3.0,Sushi,4,193,29,Card,4.0


In [146]:
data = data[data["delivery_time_minutes"] > 0]
data[data["delivery_time_minutes"] <= 0]

,order_id,customer_id,restaurant,city,order_date,delivery_time_minutes,item,quantity,price,discount,payment_method,rating


In [147]:
outliers()

['delivery_time_minutes: [69.0, 259.0, 208.0, 4.0, 3.0, 193.0, 65.0, 258.0, 73.0, 270.0, 72.0, 2.0, 68.0, 3.0, 233.0, 292.0, 178.0, 245.0, 219.0, 1.0, 68.0, 3.0, 65.0, 71.0, 72.0, 65.0]', 'quantity: [39, 22, 30, 41, 35, 25, 34, 29, 40, 28]', 'price: []', 'discount: []', 'rating: []']


There are outliers in "delivery_time_minutes" and "quantity"

In [148]:
col = "delivery_time_minutes"
Q1 = data[col].quantile(0.25)
IQR = Q3 - Q1

data[col] = data[col].clip(lower = 20)

outliers()

['delivery_time_minutes: [69.0, 259.0, 208.0, 193.0, 65.0, 258.0, 73.0, 270.0, 72.0, 68.0, 233.0, 292.0, 178.0, 245.0, 219.0, 68.0, 65.0, 71.0, 72.0, 65.0]', 'quantity: [39, 22, 30, 41, 35, 25, 34, 29, 40, 28]', 'price: []', 'discount: []', 'rating: []']


- I clipped the minutes smaller than 20 because it's impossible
- I retained the high outliers because it could be a real delay
- I retained the outliers in the quantity because it's normal

In [149]:
data["revenue"] = data["price"] * data["quantity"] - data["discount"]
data.sample()

,order_id,customer_id,restaurant,city,order_date,delivery_time_minutes,item,quantity,price,discount,payment_method,rating,revenue
1671,1672,1285,Pizza Hub,Cairo,2024-03-10 15:00:00,38.0,Pizza,2,275,16,Card,3.0,534


In [150]:
data.to_csv("data/Food_Delivery_Cleaned.csv", index = False)